# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [20]:
%pip install iso3166

Note: you may need to restart the kernel to use updated packages.


In [19]:
%pip install matplotlib seaborn plotly pandas numpy

Note: you may need to restart the kernel to use updated packages.


### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [18]:
%pip install --upgrade plotly

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 25.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.


### Import Statements

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [7]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [23]:
df_data = pd.read_csv('../data/processed/space_missions.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [24]:
df_data.shape

(5690, 7)

In [25]:
df_data.columns.tolist()

['Organisation',
 'Location',
 'Date',
 'Detail',
 'Rocket_Status',
 'Price',
 'Mission_Status']

In [26]:
df_data.isna().sum()

Organisation        0
Location            0
Date                0
Detail              0
Rocket_Status       0
Price             276
Mission_Status      0
dtype: int64

In [27]:
df_data.duplicated().sum()

np.int64(1)

## Data Cleaning - Check for Missing Values and Duplicates

Remove columns containing junk data and duplicates.

In [28]:
# Junk columns already removed in merging pipeline.

In [29]:
df_data.drop_duplicates(inplace=True)

## Descriptive Statistics

In [30]:
df_data.describe()

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
count,5689,5689,5689,5689,5689,5413,5689
unique,97,230,4943,5637,2,195,4
top,RVSN USSR,"Space Launch Complex 40, Cape Canaveral SFS, F...",2022-08-04,Titan IV(402)B | DSP,StatusRetired,12.0,Success
freq,1775,269,6,5,3974,435,5180


# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [31]:
launches_by_org = df_data["Organisation"].value_counts()
fig = px.bar(launches_by_org,
             x=launches_by_org.index,
             y=launches_by_org.values,
             title="Number of Launches by Organization")
fig.show()

In [32]:
top_20 = df_data["Organisation"].value_counts().head(20)
fig = px.bar(top_20,
              x=top_20.index,
              y=top_20.values,
              title="Top 20 Organizations by Number of Launches")
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [33]:
rocket_status = df_data["Rocket_Status"].value_counts()
fig = px.pie(rocket_status,
             values=rocket_status.values,
             names=rocket_status.index,
             title="Active vs Retired Rockets")
fig.show()

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [34]:
mission_status = df_data["Mission_Status"].value_counts()
fig = px.pie(mission_status,
             values=mission_status.values,
             names=mission_status.index,
             title="Distribution of Mission Status")
fig.show()

# How Expensive are the Launches? 

A histogram visualizing the distribution. The price column is given in USD millions (remember missing values).

In [35]:
df_data["Price"] = df_data["Price"].astype(str).str.replace(",", "").astype(float)

In [36]:
fig = px.histogram(df_data.dropna(subset=["Price"]),
                   x="Price",
                   title="Distribution of Launch Prices (USD Millions)")
fig.show()

In [37]:
fig = px.box(df_data.dropna(subset=["Price"]),
             y="Price",
             title="Distribution of Launch Prices (USD Millions)")
fig.show()

# Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

Use a 3 letter country code for each country. Might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


Use the iso3166 package to convert the country names to Alpha3 format.

In [38]:
df_data["Country"] = df_data["Location"].str.split(", ").str[-1]
print(df_data["Country"].unique())

['USA' 'China' 'Kazakhstan' 'Japan' 'Israel' 'New Zealand' 'Russia'
 'Shahrud Missile Test Site' 'France' 'Iran' 'India' 'New Mexico'
 'Yellow Sea' 'North Korea' 'Pacific Missile Range Facility'
 'Pacific Ocean' 'South Korea' 'Barents Sea' 'Brazil' 'Gran Canaria'
 'Kenya' 'Australia' 'French Guiana' "People's Republic of China"
 'Haiyang Oriental Spaceport' 'Russian Federation'
 'Republic of Kazakhstan' 'Moon' 'Air launch to Suborbital flight'
 'Air launch to orbit' 'Islamic Republic of Iran' 'South Australia'
 'Arnhem Space Centre' 'Federative Republic of Brazil' 'State of Israel'
 "Democratic People's Republic of Korea" 'El Arenosillo Test Centre'
 'Sea Launch' 'Andøya Spaceport' 'Bowen Orbital Spaceport']


In [39]:
country_replacements = {
    # From the brief
    "Russia": "Russian Federation",
    "New Mexico": "USA",
    "Yellow Sea": "China",
    "Shahrud Missile Test Site": "Iran",
    "Pacific Missile Range Facility": "USA",
    "Barents Sea": "Russian Federation",
    "Gran Canaria": "USA",
    # Additional ones
    "People's Republic of China": "China",
    "Haiyang Oriental Spaceport": "China",
    "Republic of Kazakhstan": "Kazakhstan",
    "Islamic Republic of Iran": "Iran",
    "South Australia": "Australia",
    "Arnhem Space Centre": "Australia",
    "Bowen Orbital Spaceport": "Australia",
    "Federative Republic of Brazil": "Brazil",
    "State of Israel": "Israel",
    "Democratic People's Republic of Korea": "North Korea",
    "El Arenosillo Test Centre": "Spain",
    "Andøya Spaceport": "Norway",
    # No valid country
    "Sea Launch": None,
    "Pacific Ocean": None,
    "Air launch to orbit": None,
    "Air launch to Suborbital flight": None,
    "Moon": None,
}

df_data["Country"] = df_data["Country"].replace(country_replacements)

In [41]:
df_data["Country"] = df_data["Country"].map(lambda x: country_replacements.get(x, x))
print(df_data["Country"].unique())

<StringArray>
[               'USA',              'China',         'Kazakhstan',
              'Japan',             'Israel',        'New Zealand',
 'Russian Federation',               'Iran',             'France',
              'India',        'North Korea',                  nan,
        'South Korea',             'Brazil',              'Kenya',
          'Australia',      'French Guiana',              'Spain',
             'Norway']
Length: 19, dtype: str


In [42]:
def get_alpha3(country):
    try:
        return countries.get(country).alpha3
    except:
        return None

df_data["Country_Code"] = df_data["Country"].apply(get_alpha3)
print(df_data["Country_Code"].unique())

<StringArray>
['USA', 'CHN', 'KAZ', 'JPN', 'ISR', 'NZL', 'RUS',   nan, 'FRA', 'IND', 'BRA',
 'KEN', 'AUS', 'GUF', 'ESP', 'NOR']
Length: 16, dtype: str


In [43]:
manual_codes = {
    "Iran": "IRN",
    "North Korea": "PRK",
    "South Korea": "KOR",
    "French Guiana": "FRA"
}

df_data["Country_Code"] = df_data["Country"].apply(lambda x: manual_codes.get(x) if x in manual_codes else get_alpha3(x))
print(df_data["Country_Code"].unique())

<StringArray>
['USA', 'CHN', 'KAZ', 'JPN', 'ISR', 'NZL', 'RUS', 'IRN', 'FRA', 'IND', 'PRK',
   nan, 'KOR', 'BRA', 'KEN', 'AUS', 'ESP', 'NOR']
Length: 18, dtype: str


In [44]:
country_launches = df_data["Country_Code"].value_counts().reset_index()
country_launches.columns = ["Country_Code", "Launches"]

fig = px.choropleth(country_launches,
                    locations="Country_Code",
                    color="Launches",
                    color_continuous_scale="matter",
                    title="Number of Space Launches by Country")
fig.show()

## Note on Russian/Soviet Launch Sites
The choropleth splits the Soviet and Russian launch legacy across two countries.
A significant portion of USSR and Russian launches took place from Baikonur Cosmodrome,
located in Kazakhstan. Combined, Russia (1,460) and Kazakhstan (750) account for 2,210 launches,
surpassing the USA's 2,023 and reflecting the true scale of Soviet and Russian space activity.

## Note on Kenya
Launches attributed to Kenya were conducted from the Broglio Space Centre
(formerly San Marco platform) near Malindi — an Italian-operated facility.
Kenya did not have an independent space program; these launches belong to
the Italian Space Research Commission and are counted under Kenya solely
due to their geographic location.

# Choropleth Map to Show the Number of Failures by Country


In [45]:
failures = df_data[df_data["Mission_Status"] == "Failure"]
country_failures = failures["Country_Code"].value_counts().reset_index()
country_failures.columns = ["Country_Code", "Failures"]

fig = px.choropleth(country_failures,
                    locations="Country_Code",
                    color="Failures",
                    color_continuous_scale="matter",
                    title="Number of Mission Failures by Country")
fig.show()

## Note on Failure Rates by Country
Raw failure counts can be misleading without context. The USA leads in absolute failures
(146) but has launched 2,023 missions total — a failure rate of 7.2%. Russia and Kazakhstan
combined recorded 136 failures across 2,210 launches — a failure rate of 6.2%.
The difference is marginal over nearly 70 years of launches, reflecting how both
superpowers achieved comparable reliability despite operating in vastly different
political and technological contexts.

In [48]:
country_status = df_data.groupby(["Country_Code", "Mission_Status"]).size().reset_index(name="Count")
country_totals = df_data.groupby("Country_Code").size().reset_index(name="Total")
country_status = country_status.merge(country_totals, on="Country_Code")
country_status["Percentage"] = (country_status["Count"] / country_status["Total"]) * 100

color_map = {
    "Success": "#2ecc71",
    "Failure": "#e74c3c",
    "Partial Failure": "#e67e22",
    "Prelaunch Failure": "#f1c40f"
}

fig = px.bar(country_status,
             x="Country_Code",
             y="Percentage",
             color="Mission_Status",
             color_discrete_map=color_map,
             title="Mission Success/Failure Rate by Country",
             labels={"Percentage": "% of Launches"})
fig.show()

# Sunburst Chart of the countries, organizations, and mission status.

In [50]:
sunburst_data = df_data.dropna(subset=["Country"])

fig = px.sunburst(sunburst_data,
                  path=["Country", "Organisation", "Mission_Status"],
                  title="Space Missions by Country, Organisation and Mission Status")
fig.show()

# Analysis of the Total Amount of Money Spent by Organisation on Space Missions

In [52]:
total_spend = df_data.groupby("Organisation")["Price"].sum().sort_values(ascending=False).reset_index()
total_spend["Price_Billions"] = total_spend["Price"] / 1000

fig = px.bar(total_spend,
             x="Organisation",
             y="Price_Billions",
             title="Total Amount Spent on Space Missions by Organisation (USD Billions)")
fig.show()

In [53]:
top_20_spend = total_spend.head(20)

fig = px.bar(top_20_spend,
             x="Organisation",
             y="Price_Billions",
             title="Top 20 Organisations by Total Amount Spent on Space Missions (USD Billions)")
fig.show()

# Analysis of Amount of Money Spent by Organisation per Launch

In [56]:
spend_per_launch = df_data.groupby("Organisation")["Price"].mean().sort_values(ascending=False).reset_index()
spend_per_launch["Price_Billions"] = spend_per_launch["Price"] / 1000

fig = px.bar(spend_per_launch,
             x="Organisation",
             y="Price_Billions",
             title="Average Spend per Launch by Organisation (USD Billions)")
fig.show()

In [55]:
spend_per_launch = df_data.groupby("Organisation")["Price"].mean().sort_values(ascending=False).reset_index()
spend_per_launch["Price_Billions"] = spend_per_launch["Price"] / 1000

fig = px.bar(spend_per_launch.head(20),
             x="Organisation",
             y="Price_Billions",
             title="Top 20 Organisations by Average Spend per Launch (USD Billions)")
fig.show()

In [57]:
fig = px.scatter(spend_per_launch,
                 x="Organisation",
                 y="Price_Billions",
                 title="Average Spend per Launch by Organisation (USD Billions)")
fig.show()

# Number of Launches per Year

In [58]:
df_data["Year"] = pd.to_datetime(df_data["Date"]).dt.year
launches_per_year = df_data.groupby("Year").size().reset_index(name="Launches")

fig = px.line(launches_per_year,
              x="Year",
              y="Launches",
              title="Number of Space Launches per Year")
fig.show()

## Note on 2026 Data
The sharp drop in 2026 is expected — the dataset only covers up to May 2026.
The year is not yet complete and should not be interpreted as a decline in launch activity.

# Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [59]:
df_data["Month_Year"] = pd.to_datetime(df_data["Date"]).dt.to_period("M").dt.to_timestamp()
launches_per_month = df_data.groupby("Month_Year").size().reset_index(name="Launches")
launches_per_month["Rolling_Avg"] = launches_per_month["Launches"].rolling(window=12).mean()

fig = px.line(launches_per_month,
              x="Month_Year",
              y="Launches",
              title="Number of Space Launches Month-on-Month")
fig.add_scatter(x=launches_per_month["Month_Year"],
                y=launches_per_month["Rolling_Avg"],
                name="12-Month Rolling Average")
fig.show()

## Number of Launches Month-on-Month: The Full Story

The 12-month rolling average reveals the key chapters in human spaceflight history:

**1957–1962 — Early Space Race:** Launch activity climbs rapidly as the USA and USSR
compete fiercely for dominance, peaking at ~6.8 launches per month by December 1962.

**1962–1964 — Mid-Race Dip:** A brief pullback to ~3.9 launches per month in April 1964
as both superpowers consolidated programs and transitioned to more ambitious missions.

**1964–1978 — Peak Cold War Era:** Activity surges back to ~10 launches per month and
sustains there for over a decade, driven by the Apollo program, the Soviet lunar program,
and large-scale military satellite deployments on both sides.

**1978–2015 — The Long Stagnation:** A sharp drop to ~4.3 launches per month triggered
by the US transition to the Space Shuttle — which launched far less frequently than the
expendable rockets it replaced. The 1986 Challenger disaster grounded the Shuttle fleet
for nearly 3 years, deepening the decline. Soviet economic strain under Brezhnev further
reduced cadence. Launch activity fluctuated between ~3.25 and ~6.6 for nearly four decades.

**2015–2020 — Commercial Reawakening:** SpaceX's reusable rocket technology begins
reshaping the economics of spaceflight. Activity climbs back to ~10 launches per month
by December 2018 and holds there through 2020.

**2020–Present — The Commercial Explosion:** Launch cadence explodes to 29+ per month
by 2026, driven by SpaceX's Starlink megaconstellation deployments and a thriving
commercial launch market fundamentally transforming access to space.

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [60]:
df_data["Month"] = pd.to_datetime(df_data["Date"]).dt.month
launches_per_month = df_data.groupby("Month").size().reset_index(name="Launches")
launches_per_month["Month_Name"] = pd.to_datetime(launches_per_month["Month"], format="%m").dt.strftime("%B")

fig = px.bar(launches_per_month,
             x="Month_Name",
             y="Launches",
             title="Total Number of Launches by Month")
fig.show()

## Launches by Month: Challenging the Weather Hypothesis

Conventional wisdom suggests weather conditions would drive a clear seasonal preference
for space launches, favoring spring and summer months. The data tells a more nuanced story.

**December leads** at 613 launches — not because of favourable weather, but due to the
Soviet Union's bureaucratic launch culture. End-of-year government and military program
deadlines drove a strong push to hit annual quotas before the year closed.

**January sits at the bottom** at 370 launches — post-holiday operational slowdowns,
budget cycle resets, and harsh winter conditions at high-latitude Soviet launch sites
like Baikonur in Kazakhstan all contribute.

**The remaining months are remarkably flat** ranging between 430–501 launches,
suggesting no strong global seasonal preference once the Soviet effect is accounted for.

It is worth noting that the addition of 2020–2026 data likely diluted what would have
been a sharper December peak in the original dataset. The modern commercial launch era,
led by SpaceX's very frequent Starlink missions launching year-round from Florida,
has flattened the monthly distribution significantly — making weather an increasingly
irrelevant factor in modern launch scheduling.

# Launch Price Variation Over Time?

Create a line chart that shows the average price of rocket launches over time. 

In [61]:
avg_price_per_year = df_data.groupby("Year")["Price"].mean().reset_index()
avg_price_per_year["Price_Millions"] = avg_price_per_year["Price"]

fig = px.line(avg_price_per_year,
              x="Year",
              y="Price_Millions",
              title="Average Launch Price Over Time (USD Millions)")
fig.show()

## Note on Early Era Launch Prices
Prices for Soviet-era launches were inherently secretive with the USSR not disclosing
their costs. The prices in the dataset represent educated estimates based on expert
analysis, historical records, and the economics of the Soviet command economy. Most
launches are estimated at approximately 10–12M USD with some peaks reaching 25M USD and
occasional outliers near 40M USD. Direct comparison with Western launch costs is inherently
imprecise — the Soviet economy operated on fundamentally different pricing structures
with no equivalent market mechanisms. These estimates are considered reasonable
approximations rather than verified figures, and should be interpreted accordingly
when analysing price trends prior to 1990.

## Average Launch Price Over Time: Key Trends

- **1957–1969 — Early Space Race:** Prices climb gradually from near zero,
  spiking to ~$140M around 1969 driven by the costly Apollo program —
  the most expensive missions ever flown.

- **1970s–1980s — Post-Apollo Normalization:** Prices drop back as the
  Shuttle era begins and expendable rockets dominate Soviet launches
  at estimated low costs.

- **1988–2013 — The Expensive Era:** Prices climb steadily back to ~$150M,
  reflecting the high cost of Space Shuttle operations and expensive
  Western expendable rockets with no competitive pressure.

- **2013 onwards — The SpaceX Effect:** A clear and sustained downward trend
  drops average prices to ~$50M as reusable Falcon 9 rockets disrupt
  the launch market and force competitors to reduce costs.

- **2024–2026 — Slight Uptick:** Average prices rise modestly, likely driven
  by newer high-cost programs such as NASA's Artemis pushing the
  average back up despite continued commercial competition.

# Number of Launches over Time by the Top 10 Organizations.

How has the dominance of launches changed over time between the different players? 

In [62]:
top_10_orgs = df_data["Organisation"].value_counts().head(10).index

df_top10 = df_data[df_data["Organisation"].isin(top_10_orgs)]
launches_by_org_year = df_top10.groupby(["Year", "Organisation"]).size().reset_index(name="Launches")

fig = px.line(launches_by_org_year,
              x="Year",
              y="Launches",
              color="Organisation",
              title="Number of Launches Over Time by Top 10 Organizations")
fig.show()

## Number of Launches Over Time by Top 10 Organizations

The chart tells the definitive story of launch dominance across nearly 70 years of spaceflight:

- **RVSN USSR** dominated from the late 1950s through the late 1980s, peaking at 97 launches
  in a single year. Its collapse was as sudden as it was dramatic — the dissolution of the
  Soviet Union in 1991 brought Soviet launch activity to a near standstill almost overnight.

- **Everyone else remained negligible for decades** — NASA, the US Air Force, General Dynamics,
  and Arianespace all flatline in comparison to the Soviet industrial-scale launch machine.

- **SpaceX** doesn't appear until the 2010s, then rewrites the record books — climbing to an
  extraordinary 170 launches in 2025 alone, nearly double the USSR's all-time annual peak.
  No organization in history has come close to this cadence.

- **CASC (China)** begins a quiet but steady climb from 2015 onwards, establishing itself
  as a serious third player in the modern launch landscape.

- **Note:** The visible drop in SpaceX's 2026 count reflects an incomplete year —
  data only covers through May 2026.

The handoff of dominance from RVSN USSR to SpaceX is the single most defining visual
story of this entire dataset — separated by three decades but connected by the same
relentless pursuit of launch volume.

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

In [63]:
cold_war = df_data[pd.to_datetime(df_data["Date"]).dt.year <= 1991].copy()

usa = cold_war[cold_war["Country_Code"] == "USA"]
ussr = cold_war[cold_war["Country_Code"].isin(["RUS", "KAZ"])]

print(f"USA launches: {len(usa)}")
print(f"USSR launches: {len(ussr)}")

USA launches: 662
USSR launches: 1768


In [65]:
cold_war_totals = pd.DataFrame({
    "Superpower": ["USA", "USSR"],
    "Launches": [len(usa), len(ussr)]
})

fig = px.pie(cold_war_totals,
             values="Launches",
             names="Superpower",
             title="Total Launches During the Cold War (1957-1991)",
             color="Superpower",
             color_discrete_map={
                 "USA": "#3a5f8a",
                 "USSR": "#8b2020"
             })
fig.show()

In [66]:
usa_by_year = usa.groupby("Year").size().reset_index(name="Launches")
usa_by_year["Superpower"] = "USA"

ussr_by_year = ussr.groupby("Year").size().reset_index(name="Launches")
ussr_by_year["Superpower"] = "USSR"

cold_war_by_year = pd.concat([usa_by_year, ussr_by_year])

fig = px.line(cold_war_by_year,
              x="Year",
              y="Launches",
              color="Superpower",
              color_discrete_map={
                  "USA": "#3a5f8a",
                  "USSR": "#8b2020"
              },
              title="Cold War Space Race: USA vs USSR Launches Year-on-Year (1957-1991)")
fig.show()

## Cold War Space Race: USA vs USSR Year-on-Year (1957-1991)

**1957-1963 — Early American Lead:** The USA held the early advantage,
peaking at ~60 launches in 1962 while the USSR was still ramping up.
The race was genuinely competitive in these opening years.

**1963-1966 — The Crossover:** The USSR overtook the USA decisively around
1963 and never looked back. The USA briefly surged to 48 launches in 1966
— likely driven by the Apollo program buildup — but it was the last time
America would seriously challenge Soviet launch volume.

**1966-1978 — Soviet Dominance:** The USSR pulled away dramatically,
sustaining 79-97 launches per year while the USA stagnated between just
9 and 21. This was the height of Soviet industrial-scale spaceflight —
military satellites, crewed missions, and planetary probes launching
relentlessly.

**1977-1979 — Soviet Pullback:** USSR launches dropped sharply from their
peak to 36 per year. Economic pressures and program consolidations under
the Brezhnev era began to slow the machine.

**1979-1989 — Sustained but Declining Soviet Lead:** The USSR stabilized
between 36-53 launches annually while the USA remained flat. The gap
narrowed but Soviet dominance remained unchallenged throughout the 1980s.

**1990-1991 — The Collapse:** Both countries dropped sharply as the Soviet
Union dissolved. The Cold War Space Race ended not with a climax but with
the quiet implosion of one of its two protagonists.

In [67]:
usa_by_year["Cumulative"] = usa_by_year["Launches"].cumsum()
ussr_by_year["Cumulative"] = ussr_by_year["Launches"].cumsum()

cold_war_cumulative = pd.concat([usa_by_year, ussr_by_year])

fig = px.line(cold_war_cumulative,
              x="Year",
              y="Cumulative",
              color="Superpower",
              color_discrete_map={
                  "USA": "#3a5f8a",
                  "USSR": "#8b2020"
              },
              title="Cold War Space Race: Cumulative Launches USA vs USSR (1957-1991)")
fig.show()

## Cold War Space Race: Cumulative Launches (1957-1991)

The cumulative chart strips away the noise and reveals the definitive story
of the Cold War Space Race in a single image.

**1957-1963 — Neck and Neck:** Both superpowers started evenly matched,
with cumulative totals tracking closely in the early years of the race.

**1963 — The Decisive Split:** The USSR pulled ahead permanently around 1963
and the gap never closed. From this point forward the Soviet launch machine
operated at a scale the USA simply never matched.

**1963-1991 — Relentless Soviet Expansion:** The USSR's cumulative line
accelerates away in a near-straight trajectory, reflecting the sustained
industrial-scale launch cadence that defined Soviet space strategy.
Volume was the Soviet doctrine — launch more, launch often.

**By 1991 — The Final Tally:** The Cold War ended with the USSR at 1,768
cumulative launches against the USA's 662 — the Soviets launched
nearly 3x more missions over the course of the entire race.

The USA won the Moon. The USSR won the volume war. Both facts are visible
in this dataset.

In [68]:
usa_status = usa["Mission_Status"].value_counts().reset_index()
usa_status.columns = ["Mission_Status", "Count"]
usa_status["Superpower"] = "USA"

ussr_status = ussr["Mission_Status"].value_counts().reset_index()
ussr_status.columns = ["Mission_Status", "Count"]
ussr_status["Superpower"] = "USSR"

cold_war_status = pd.concat([usa_status, ussr_status])

fig = px.bar(cold_war_status,
             x="Superpower",
             y="Count",
             color="Mission_Status",
             barmode="group",
             color_discrete_map={
                 "Success": "#2ecc71",
                 "Failure": "#e74c3c",
                 "Partial Failure": "#e67e22",
                 "Prelaunch Failure": "#f1c40f"
             },
             title="Mission Success vs Failure: USA vs USSR During the Cold War (1957-1991)")
fig.show()

## Cold War Mission Success Rates: USA vs USSR (1957-1991)

Perhaps the most surprising finding of the entire Cold War analysis — the USSR not only
launched nearly 3x more missions than the USA, but did so with a significantly higher
success rate.

**USA:** 536 successes, 101 failures, 25 partial failures — an overall success rate of 80.97%

**USSR:** 1,606 successes, 120 failures, 41 partial failures — an overall success rate of 90.84%

The USSR's superior reliability despite its overwhelming launch volume challenges the
Western narrative of Soviet space technology as rushed or reckless. The data suggests
a highly disciplined and mature launch operation at its peak.

The USA's comparatively higher failure rate likely reflects a greater appetite for
experimental and boundary-pushing missions — the Apollo program, deep space probes,
and novel spacecraft designs that carried inherently higher risk. The USSR by contrast
favored proven, repeatable rocket designs launched at industrial scale — a philosophy
that clearly paid dividends in reliability.

The USA won the Moon. The USSR won on volume and reliability. The Space Race was
far more nuanced than history often remembers.

In [69]:
cold_war_cumulative_reset = cold_war_cumulative.reset_index(drop=True)

fig = px.bar(cold_war_cumulative_reset,
             x="Superpower",
             y="Cumulative",
             color="Superpower",
             animation_frame="Year",
             range_y=[0, 1800],
             color_discrete_map={
                 "USA": "#3a5f8a",
                 "USSR": "#8b2020"
             },
             title="Cold War Space Race: Cumulative Launches USA vs USSR (1957-1991)")
fig.show()

## Cold War Space Race: Section Summary

The numbers tell one story — the USSR won the volume war decisively, launching nearly
3x more missions than the USA with a higher success rate. But raw launch counts alone
cannot capture the full legacy of the Cold War Space Race.

**The USSR owned the early race and pioneered human spaceflight:**
- **Sputnik 1 (1957):** First satellite ever launched, shocking the Western world
- **Yuri Gagarin (1961):** First human in space
- **Valentina Tereshkova (1963):** First woman in space
- **Alexei Leonov (1965):** First spacewalk
- **Venera 7 (1970):** First successful landing on another planet (Venus)
- **Mir Space Station:** First long-duration space station, foundational to our
  understanding of long-term human spaceflight
- **Soyuz rocket:** Still flying today — one of the most reliable launch vehicles
  ever built, a testament to Soviet engineering discipline

**The USA responded with deeper ambition and broader scientific reach:**
- **Apollo Program (1961-1972):** 12 humans walked on the Moon across 6 missions —
  arguably the single greatest achievement in human exploration
- **Voyager 1 & 2 (1977):** Still the most distant human-made objects ever launched,
  now traveling through interstellar space
- **Mariner Program:** First successful flybys of Mars, Venus, and Mercury —
  America mapped the inner solar system
- **Viking Mars Landers (1976):** First successful soft landing and surface
  operations on Mars
- **Hubble Space Telescope (1990):** Fundamentally transformed humanity's
  understanding of the universe

**The bigger picture:**
The Cold War Space Race was driven by rivalry but powered by human curiosity.
The USSR launched more, the USA reached farther — but both superpowers pushed
the boundaries of what was possible in ways that continue to define spaceflight today.
Every mission that followed, from the International Space Station to Artemis,
stands on the shoulders of what these two nations built between 1957 and 1991.

## Total Number of Mission Failures Year on Year.

In [70]:
failures_by_year = df_data[df_data["Mission_Status"] == "Failure"].groupby("Year").size().reset_index(name="Failures")

fig = px.line(failures_by_year,
              x="Year",
              y="Failures",
              title="Total Number of Mission Failures Year-on-Year")
fig.show()

## Total Number of Mission Failures Year-on-Year

The decline in annual failures tells the story of an industry that learned,
adapted, and matured over nearly 70 years of spaceflight.

**Late 1950s-1960s — The Experimental Era:** Failures peaked at 20 in 1958 and 1960
as both superpowers pushed untested technology to its limits. Rockets were
essentially experimental vehicles and failure was an expected part of the
process. Engineers learned more from failures than successes in these
foundational years.

**1960s-1980s — The Learning Curve:** A steady and sustained decline in
failures as both the USA and USSR refined their designs, standardized their
processes, and accumulated operational experience. The science of rocketry
was maturing rapidly.

**1980s-2015 — The Reliable Era:** Failures flatlined at just 1-5 per year
— a remarkable achievement considering the complexity of spaceflight.
Proven rocket designs like the Soyuz, Atlas, and Ariane dominated the
launch manifest, minimizing risk through repetition and refinement.

**2020s — The New Player Effect:** A modest uptick in raw failures reflects
the explosion in launch volume rather than a decline in reliability.
New commercial players and emerging space nations are entering the market
with less mature vehicles, mirroring the experimental spirit of the 1960s
at a fraction of the risk. The failure *rate* remains historically low
even as raw numbers tick upward.

## Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [71]:
total_by_year = df_data.groupby("Year").size().reset_index(name="Total")
failures_by_year = df_data[df_data["Mission_Status"] == "Failure"].groupby("Year").size().reset_index(name="Failures")

failure_rate = total_by_year.merge(failures_by_year, on="Year", how="left").fillna(0)
failure_rate["Failure_Pct"] = (failure_rate["Failures"] / failure_rate["Total"]) * 100

fig = px.line(failure_rate,
              x="Year",
              y="Failure_Pct",
              title="Percentage of Mission Failures Over Time")
fig.show()

## Percentage of Mission Failures Over Time

This chart answers the question definitively — yes, spaceflight got dramatically
safer over time.

**1957 — The Experimental Peak:** A staggering 71.43% failure rate in the very
first year of the Space Age. More than 2 in every 3 launches failed. Rockets were
experimental vehicles and failure was not the exception — it was the norm.

**1957-1975 — The Learning Curve:** A sharp and sustained decline in failure rates
as both superpowers accumulated experience, refined their designs, and standardized
their operations. By 1975 the failure rate had dropped to just 5.31% — a
transformation achieved in less than two decades.

**1975-2019 — The Reliable Era:** Failure rates fluctuated in a remarkably tight
band, ranging from a low of 1.03% in 1978 to a high of 10.53% in 1999.
Spaceflight had become a mature, repeatable engineering discipline.

**2020-2026 — The Commercial Era:** The explosion in launch volume introduced
new players and newer vehicles, pushing the upper band to 8.26% in 2020.
However the trend quickly corrected — dropping to a low of 2.20%
in 2024, 3.52% in 2025, and sitting at 5.45% in 2026. The industry absorbed
the volume shock without a meaningful long-term deterioration in reliability.

The data is unambiguous — humanity went from failing 71% of launches in 1957
to failing less than 4% on average in the modern era. That is one of the most
remarkable engineering achievements in human history.

# Country Lead in terms of Total Number of Launches (per year) up to and including 2026

Do the results change if we only look at the number of successful launches? 

In [73]:
launches_by_country_year = df_data[df_data["Year"] <= 2026].groupby(["Year", "Country_Code"]).size().reset_index(name="Launches")
leading_country = launches_by_country_year.loc[launches_by_country_year.groupby("Year")["Launches"].idxmax()]

color_map = {
    "USA": "#3a5f8a",
    "RUS": "#8b2020",
    "KAZ": "#c0392b",
    "CHN": "#f39c12"
}

fig = px.bar(leading_country,
             x="Year",
             y="Launches",
             color="Country_Code",
             color_discrete_map=color_map,
             title="Country with the Most Launches Per Year")
fig.show()

In [74]:
successful = df_data[df_data["Mission_Status"] == "Success"]
success_by_country_year = successful[successful["Year"] <= 2026].groupby(["Year", "Country_Code"]).size().reset_index(name="Launches")
leading_success = success_by_country_year.loc[success_by_country_year.groupby("Year")["Launches"].idxmax()]

fig = px.bar(leading_success,
             x="Year",
             y="Launches",
             color="Country_Code",
             color_discrete_map=color_map,
             title="Country with the Most Successful Launches Per Year")
fig.show()

## Leading Country by Launches Per Year (1957-2026)

**Total Launches:** The USA led the early Space Race through the late 1950s
before the USSR took over decisively from the early 1960s. China began asserting itself from
2018 onward before the USA reclaimed dominance driven almost entirely
by SpaceX's commercial launch explosion.

**Successful Launches Only:** The results are virtually identical — the
dominant country each year remains mostly unchanged when filtering to successful
missions only. The early era shows shorter bars reflecting the higher
failure rates of experimental spaceflight, but the leadership picture
does not change. Volume and reliability tracked together — the country
launching the most was also succeeding the most.

# Year-on-Year Chart Showing the Organization Doing the Most Number of Launches

Which organization was dominant in the 1970s and 1980s? Which organization was dominant in 2018, 2019 and 2020? Lastly, which organization was dominant in 2024, 2025, and 2026?

In [76]:
launches_by_org_year = df_data.groupby(["Year", "Organisation"]).size().reset_index(name="Launches")
leading_org = launches_by_org_year.loc[launches_by_org_year.groupby("Year")["Launches"].idxmax()]

color_map = {
    "RVSN USSR": "#8b2020",
    "US Navy": "#003087",
    "US Air Force": "#5482ab",
    "VKS RF": "#c0392b",
    "Lockheed": "#1a3a5c",
    "Arianespace": "#005bac",
    "Boeing": "#0033a0",
    "ULA": "#1b3a6b",
    "CASC": "#de2910",
    "China Aerospace Science and Technology Corporation": "#de2910",
    "SpaceX": "#2c3e50"
}

fig = px.bar(leading_org,
             x="Year",
             y="Launches",
             color="Organisation",
             color_discrete_map=color_map,
             title="Organization with the Most Launches Per Year")
fig.show()

## Organization with the Most Launches Per Year

**1970s and 1980s — RVSN USSR Dominance:** The Soviet Strategic Rocket Forces
were untouchable for nearly three decades, peaking at 97 launches in a single
year. No other organization came remotely close during this era.

**A Note on NASA:** Notably absent from this chart despite being the world's most
recognized space agency. NASA is primarily a research and development organization
— they design missions and payloads but contract the actual launches to other
organizations. NASA's missions appear under US Air Force, Lockheed, Boeing, ULA,
and SpaceX depending on the era. NASA's influence on spaceflight far exceeds
what any launch count could capture.

**2018-2019 — China's Moment:** CASC (China Aerospace Science and Technology
Corporation) briefly seized the lead with 37 launches in 2018 and 27 in 2019 —
signaling China's emergence as a genuine spacefaring superpower and its
ambitions beyond Earth orbit.

**2020 — The Handoff:** SpaceX took the lead for the first time with 29 launches,
marking the beginning of an era of American commercial dominance that
shows no signs of slowing.

**2024-2026 — SpaceX Rewrites History:** SpaceX's dominance becomes almost
incomprehensible in scale — 138 launches in 2024, 170 in 2025, and already
55 through May 2026. To put this in perspective, SpaceX in 2025 alone launched
nearly double the USSR's all-time annual peak of 97. No organization in the
history of spaceflight has ever approached this cadence.